In [1]:
import os
import ctypes
import sys
from pathlib import Path

# --- 1. PATH native library setup ---
root = Path.cwd()

if sys.platform.startswith('linux'):
    lib_dir = root / 'pathlib' / 'lib_lnx'
    lib_name = 'pathwrap.so'
    path_lib_name = 'libpath50.so'
elif sys.platform == 'darwin':
    lib_dir = root / 'pathlib' / 'lib_osx'
    lib_name = 'pathwrap.dylib'
    path_lib_name = 'libpath50.dylib'
elif sys.platform.startswith('win'):
    lib_dir = root / 'pathlib' / 'lib_win'
    lib_name = 'pathwrap.dll'
    path_lib_name = 'libpath50.dll'
else:
    lib_dir = root / 'pathlib' / 'lib_lnx'
    lib_name = 'pathwrap.so'
    path_lib_name = 'libpath50.so'

pathwrap_path = root / lib_name

if not pathwrap_path.exists():
    print('Missing pathwrap library:', pathwrap_path)
    print('Build it with:')
    print('  gcc -shared -fPIC -Ipathlib/include -Ipathlib/examples/C -o pathwrap.so \\')
    print('    pathwrap.c pathlib/examples/C/Standalone_Path.c \\')
    print('    -Lpathlib/lib_lnx -lpath50 -lm -ldl')

# Preload PATH shared library BEFORE anything tries to use pathwrap
if lib_dir.exists():
    ld_path = os.environ.get('LD_LIBRARY_PATH', '')
    if str(lib_dir) not in ld_path:
        os.environ['LD_LIBRARY_PATH'] = f"{lib_dir}:{ld_path}" if ld_path else str(lib_dir)

if not os.environ.get('PATH_LICENSE_STRING'):
    os.environ['PATH_LICENSE_STRING'] = '1259252040&Courtesy&&&USR&GEN2035&5_1_2026&1000&PATH&GEN&31_12_2035&0_0_0&6000&0_0'


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from GridWorld import GridWorldEnv
from SRQagent import SRQAgent

In [ ]:
def train_experiment(n_episodes=3000, p_env=0.8, pathwrap_path=None, checkpoint_interval=200, checkpoint_dir="checkpoints_scenario1"):

    env = GridWorldEnv(p=p_env)

    pathwrap_path = pathwrap_path or str(globals().get("pathwrap_path", "pathwrap.so"))

    num_agents = 2
    num_actions = len(env.action_space)

    agents = [
        SRQAgent(
            agent_id=0,
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            alpha=0.1,
            gamma=0.9,
            decay_rate=0.998,
            pathwrap_path=pathwrap_path
        ),
        SRQAgent(
            agent_id=1,
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            alpha=0.1,
            gamma=0.9,
            decay_rate=0.998,
            pathwrap_path=pathwrap_path
        )
    ]

    history_rewards = [[], []]
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    best_joint_reward = -float("inf")

    print(f"Starting Training: {n_episodes} Episodes, p={p_env}")

    for ep in tqdm(range(n_episodes)):
        obs = env.reset()
        done = False
        ep_rewards = [0, 0]

        while not done:
            shared_policies = SRQAgent.solve_shared_sre(agents, obs)
            actions = [ag.act(obs, policies=shared_policies) for ag in agents]
            next_obs, rewards, done, _ = env.step(actions)
            next_shared_policies = SRQAgent.solve_shared_sre(agents, next_obs)

            for ag in agents:
                ag.update(obs, actions, rewards, next_obs, done=done, next_policies=next_shared_policies)

            obs = next_obs
            ep_rewards[0] += rewards[0]
            ep_rewards[1] += rewards[1]

        history_rewards[0].append(ep_rewards[0])
        history_rewards[1].append(ep_rewards[1])

        if (ep + 1) % checkpoint_interval == 0:
            for idx, ag in enumerate(agents):
                ag.save_q_table(checkpoint_path / f"srq_agent{idx}_ep{ep + 1}.pkl")

        joint_reward = ep_rewards[0] + ep_rewards[1]
        if joint_reward > best_joint_reward:
            best_joint_reward = joint_reward
            for idx, ag in enumerate(agents):
                ag.save_q_table(checkpoint_path / f"srq_agent{idx}_best.pkl")

        for ag in agents:
            ag.decay_parameters()

    return history_rewards

rewards_scenario1 = train_experiment(n_episodes=3000, p_env=0.8)

In [4]:
import pickle

def build_training_stats(
    scenario_name,
    rewards,
    *,
    n_episodes,
    p_env,
    grid_size,
    start_positions,
    goal_positions,
    window_size=25,
    separator=10,
    last_n=1000,
):
    return {
        "scenario_name": scenario_name,
        "rewards": [[float(reward) for reward in agent_rewards] for agent_rewards in rewards],
        "n_episodes": int(n_episodes),
        "p_env": float(p_env),
        "grid_size": int(grid_size),
        "start_positions": [tuple(position) for position in start_positions],
        "goal_positions": [tuple(position) for position in goal_positions],
        "window_size": int(window_size),
        "separator": int(separator),
        "last_n": int(last_n),
    }

def save_training_stats(stats_path, scenario_name, rewards, **metadata):
    stats = build_training_stats(scenario_name, rewards, **metadata)
    with open(stats_path, "wb") as f:
        pickle.dump(stats, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Saved training stats to {stats_path}")
    return stats

scenario1_stats = save_training_stats(
    "training_stats_scenario1.pkl",
    "Scenario 1",
    rewards_scenario1,
    n_episodes=3000,
    p_env=0.8,
    grid_size=3,
    start_positions=[(2, 0), (2, 2)],
    goal_positions=[(0, 2), (0, 0)],
)

## Scenario 2: 3×3 Grid — Agent 1 Top-Left → Bottom-Right, Agent 2 Bottom-Right → Top-Left

In [ ]:
def train_experiment_custom(
    n_episodes=3000,
    p_env=0.8,
    grid_size=3,
    start_positions=None,
    goal_positions=None,
    pathwrap_path=None,
    checkpoint_interval=200,
    checkpoint_dir="checkpoints",
):
    env = GridWorldEnv(
        grid_size=grid_size,
        p=p_env,
        start_positions=start_positions,
        goal_positions=goal_positions,
    )

    pathwrap_path = pathwrap_path or str(globals().get("pathwrap_path", "pathwrap.so"))

    num_agents = 2
    num_actions = len(env.action_space)

    agents = [
        SRQAgent(
            agent_id=0,
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            alpha=0.1,
            gamma=0.9,
            decay_rate=0.998,
            pathwrap_path=pathwrap_path,
        ),
        SRQAgent(
            agent_id=1,
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            alpha=0.1,
            gamma=0.9,
            decay_rate=0.998,
            pathwrap_path=pathwrap_path,
        ),
    ]

    history_rewards = [[], []]
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    best_joint_reward = -float("inf")

    print(
        f"Starting Training: {n_episodes} Episodes, grid={grid_size}x{grid_size}, p={p_env}, "
        f"starts={env.start_positions}, goals={env.goal_positions}"
    )

    for ep in tqdm(range(n_episodes)):
        obs = env.reset()
        done = False
        ep_rewards = [0, 0]

        while not done:
            shared_policies = SRQAgent.solve_shared_sre(agents, obs)
            actions = [ag.act(obs, policies=shared_policies) for ag in agents]
            next_obs, rewards, done, _ = env.step(actions)
            next_shared_policies = SRQAgent.solve_shared_sre(agents, next_obs)

            for ag in agents:
                ag.update(obs, actions, rewards, next_obs, done=done, next_policies=next_shared_policies)

            obs = next_obs
            ep_rewards[0] += rewards[0]
            ep_rewards[1] += rewards[1]

        history_rewards[0].append(ep_rewards[0])
        history_rewards[1].append(ep_rewards[1])

        if (ep + 1) % checkpoint_interval == 0:
            for idx, ag in enumerate(agents):
                ag.save_q_table(checkpoint_path / f"srq_agent{idx}_ep{ep + 1}.pkl")

        joint_reward = ep_rewards[0] + ep_rewards[1]
        if joint_reward > best_joint_reward:
            best_joint_reward = joint_reward
            for idx, ag in enumerate(agents):
                ag.save_q_table(checkpoint_path / f"srq_agent{idx}_best.pkl")

        for ag in agents:
            ag.decay_parameters()

    return history_rewards


# Scenario 2: 3x3, Agent 1 top-left (0,0) -> bottom-right (2,2),
#                   Agent 2 bottom-right (2,2) -> top-left (0,0)
rewards_scenario2 = train_experiment_custom(
    n_episodes=3000,
    p_env=0.8,
    grid_size=3,
    start_positions=[(0, 0), (2, 2)],
    goal_positions=[(2, 2), (0, 0)],
    checkpoint_dir="checkpoints_scenario2",
)

scenario2_stats = save_training_stats(
    "training_stats_scenario2.pkl",
    "Scenario 2",
    rewards_scenario2,
    n_episodes=3000,
    p_env=0.8,
    grid_size=3,
    start_positions=[(0, 0), (2, 2)],
    goal_positions=[(2, 2), (0, 0)],
)

## Scenario 3: 4×4 Grid — Agent 1 Bottom-Left → Top-Right, Agent 2 Bottom-Right → Top-Left

In [ ]:
# Scenario 3: 4x4, Agent 1 bottom-left (3,0) -> top-right (0,3),
#                   Agent 2 bottom-right (3,3) -> top-left (0,0)
rewards_scenario3 = train_experiment_custom(
    n_episodes=3000,
    p_env=0.8,
    grid_size=4,
    start_positions=[(3, 0), (3, 3)],
    goal_positions=[(0, 3), (0, 0)],
    checkpoint_dir="checkpoints_scenario3",
)

scenario3_stats = save_training_stats(
    "training_stats_scenario3.pkl",
    "Scenario 3",
    rewards_scenario3,
    n_episodes=3000,
    p_env=0.8,
    grid_size=4,
    start_positions=[(3, 0), (3, 3)],
    goal_positions=[(0, 3), (0, 0)],
)

In [ ]:
from dueling_double_dqn_sre import DuelingDoubleDqnSreAgent
import torch

def _flatten_obs(obs):
    return np.array([coord for pos in obs for coord in pos], dtype=np.float32)

def train_dueling_double_experiment(
    n_episodes=1000,
    grid_size=4,
    p_env=0.8,
    pathwrap_path=None,
    batch_size=32,
    target_update=10,
    checkpoint_interval=100,
    checkpoint_dir="checkpoints_dueling_sre_ddqn",
    use_gpu=True,
):
    env = GridWorldEnv(grid_size=grid_size, p=p_env)
    pathwrap_path = pathwrap_path or str(globals().get("pathwrap_path", "pathwrap.so"))

    num_agents = 2
    num_actions = len(env.action_space)
    obs_dim = len(_flatten_obs(env.reset()))

    agents = [
        DuelingDoubleDqnSreAgent(
            agent_id=0,
            obs_dim=obs_dim,
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            lr=1e-3,
            gamma=0.9,
            decay_rate=0.998,
            buffer_size=10000,
            pathwrap_path=pathwrap_path,
            use_gpu=use_gpu,
        ),
        DuelingDoubleDqnSreAgent(
            agent_id=1,
            obs_dim=obs_dim,
            num_agents=num_agents,
            num_actions=num_actions,
            epsilon_robust=1.0,
            epsilon_explore=1.0,
            lr=1e-3,
            gamma=0.9,
            decay_rate=0.998,
            buffer_size=10000,
            pathwrap_path=pathwrap_path,
            use_gpu=use_gpu,
        ),
    ]

    history_rewards = [[], []]
    checkpoint_path = Path(checkpoint_dir)
    checkpoint_path.mkdir(parents=True, exist_ok=True)

    print(
        f"Starting Dueling Double DQN-SRE Training: {n_episodes} Episodes, "
        f"grid={grid_size}x{grid_size}, p={p_env}"
    )

    for ep in tqdm(range(n_episodes)):
        obs = env.reset()
        done = False
        ep_rewards = [0.0, 0.0]

        while not done:
            actions = [ag.act(obs) for ag in agents]
            next_obs, rewards, done, _ = env.step(actions)

            obs_vec = _flatten_obs(obs)
            next_obs_vec = _flatten_obs(next_obs)

            for idx, ag in enumerate(agents):
                ag.update(
                    state=obs_vec,
                    joint_actions=actions,
                    joint_rewards=rewards,
                    next_state=next_obs_vec,
                    done=done,
                    batch_size=batch_size,
                )
                ep_rewards[idx] += float(rewards[idx])

            obs = next_obs

        history_rewards[0].append(ep_rewards[0])
        history_rewards[1].append(ep_rewards[1])

        if (ep + 1) % target_update == 0:
            for ag in agents:
                ag.update_target_network()

        if (ep + 1) % checkpoint_interval == 0:
            for idx, ag in enumerate(agents):
                ag.save_checkpoint(checkpoint_path / f"dueling_sre_agent{idx}_ep{ep + 1}.pt")

        for ag in agents:
            ag.decay_parameters()

    for idx, ag in enumerate(agents):
        ag.save_checkpoint(checkpoint_path / f"dueling_sre_agent{idx}_final.pt")

    return history_rewards, agents

dueling_rewards, dueling_agents = train_dueling_double_experiment(
    n_episodes=1000,
    grid_size=4,
    p_env=0.8,
    pathwrap_path=pathwrap_path,
    use_gpu=torch.cuda.is_available(),
)

OSError: libpath50.so: cannot open shared object file: No such file or directory

In [ ]:
import pickle

# Save reward histories
with open("history_rewards_dueling_double_sre.pkl", "wb") as f:
    pickle.dump(dueling_rewards, f, protocol=pickle.HIGHEST_PROTOCOL)

## Comparison Harness

This section mirrors Jack's pairwise-comparison workflow. It trains the six unordered pairings across the three grid scenarios and saves one `training_stats.pkl` artifact per run under `comparison_runs/<scenario>/<pair>/`.

In [ ]:
from comparison_harness import DEFAULT_PAIRINGS, SCENARIO_CONFIGS, run_all_pairings

comparison_pairings = DEFAULT_PAIRINGS
comparison_scenarios = list(SCENARIO_CONFIGS.keys())
comparison_pairings

In [ ]:
try:
    import torch
    comparison_use_gpu = torch.cuda.is_available()
except ImportError:
    comparison_use_gpu = False

comparison_results = run_all_pairings(
    scenarios=comparison_scenarios,
    pairings=comparison_pairings,
    n_episodes=3000,
    pathwrap_path=pathwrap_path,
    output_root="comparison_runs",
    use_gpu=comparison_use_gpu,
)

comparison_results